# 01 — Exploratory analysis (midterm figures)

Figures saved to `figures/`:
1. County map of outage frequency
2. Outage severity by storm type
3. Distribution of restoration times
4. SVI vs restoration time (first look)
5. Case studies: Texas Feb 2021 winter storm, Hurricane Ida (Aug 2021)

In [ ]:
# Works on Colab (Drive) or locally (repo checkout)
import os, sys, pathlib, warnings
warnings.filterwarnings("ignore")
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = pathlib.Path("/content/drive/MyDrive/LightsOut")
    ZIP = pathlib.Path("/content/drive/MyDrive/LightsOut_code.zip")
    if ZIP.exists():  # refresh code (src/, notebooks/) from the latest uploaded zip; data/ is untouched
        import zipfile; zipfile.ZipFile(ZIP).extractall(PROJECT)
    !pip -q install polars lifelines scikit-survival shap geopandas lightgbm
except ImportError:
    PROJECT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(PROJECT / "src"))
os.chdir(PROJECT)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
from lightsout import config as C, viz
viz.setup()
df = pd.read_parquet(C.PROCESSED / "model_table.parquet")
print(f"{len(df):,} (storm episode, county) units, {df.fips.nunique():,} counties, years {df.year.min()}-{df.year.max()}")

In [ ]:
ev = pd.concat(pd.read_parquet(p) for p in sorted(C.INTERIM.glob("outage_events_*.parquet")))
daily = pd.concat(pd.read_parquet(p).assign(year=int(p.stem[-4:])) for p in sorted(C.INTERIM.glob("daily_*.parquet")))
print(f"{len(ev):,} outage events detected in EAGLE-I")
summary = df.groupby("year").agg(units=("fips", "size"), counties=("fips", "nunique"),
                                 outage_rate=("had_outage", "mean"),
                                 median_peak_pct_if_outage=("peak_pct", lambda s: 100 * s[s > 0].median()))
summary.round(3)

## 1. Where do outages happen? (outage events per county per year)

In [ ]:
import geopandas as gpd
counties = gpd.read_file(C.RAW / "geo" / "counties" / "cb_2020_us_county_20m.shp")
counties["fips"] = counties["GEOID"].astype(int)
counties = counties[~counties["STATEFP"].isin(["02", "15", "72"])].to_crs(5070)  # contiguous US

years_covered = daily.groupby("fips")["year"].nunique()
freq = (ev.groupby("fips").size() / years_covered).rename("events_per_year")
g = counties.merge(freq, left_on="fips", right_index=True, how="left")

fig, ax = plt.subplots(figsize=(11, 6.5))
g.plot(column="events_per_year", ax=ax, cmap=viz.SEQUENTIAL_CMAP, scheme="quantiles", k=5,
       legend=True, legend_kwds={"title": "Outage events / year", "loc": "lower right", "fmt": "{:.1f}"},
       missing_kwds={"color": viz.NO_DATA, "hatch": "////", "edgecolor": viz.SURFACE, "label": "No EAGLE-I data"}, edgecolor="#fcfcfb", linewidth=0.1)
ax.set_axis_off()
ax.set_title("Outage events per county per year (EAGLE-I, events ≥ max(100 customers, 0.5%))")
viz.save(fig, "01_outage_frequency_map"); fig

## 2. Severity by storm type

In [ ]:
order = (df[df.had_outage].groupby("storm_type")["peak_pct"].median().sort_values(ascending=False).index)
order = [t for t in order if (df.storm_type == t).sum() >= 50]

fig, ax = plt.subplots(figsize=(9, 4.5))
rate = df.groupby("storm_type")["had_outage"].mean().reindex(order) * 100
ax.bar(rate.index, rate.values, color=[viz.storm_color(t) for t in rate.index], width=0.6)
for i, v in enumerate(rate.values):
    ax.text(i, v + 0.5, f"{v:.0f}%", ha="center", color=viz.INK_2, fontsize=9)
ax.set_ylabel("% of storm-county units with an outage")
ax.set_title("How often each storm type causes a significant outage")
viz.save(fig, "02a_outage_rate_by_storm_type"); fig

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
data = [100 * df.loc[df.had_outage & (df.storm_type == t), "peak_pct"] for t in order]
bp = ax.boxplot(data, tick_labels=order, showfliers=False, patch_artist=True, widths=0.55,
                medianprops={"color": viz.INK, "linewidth": 1.5})
for patch, t in zip(bp["boxes"], order):
    patch.set_facecolor(viz.storm_color(t)); patch.set_edgecolor(viz.SURFACE)
ax.set_yscale("log"); ax.set_ylabel("Peak % of customers out (log scale)")
ax.set_title("Peak outage severity by storm type (units with an outage)")
viz.save(fig, "02b_severity_by_storm_type"); fig

## 3. Restoration times

In [ ]:
from lifelines import KaplanMeierFitter
s = df[df.had_outage]
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].hist(s.loc[s.restored == 1, "restore_hours"].clip(0.25), bins=np.logspace(-0.6, 2.6, 40),
             color=viz.CATEGORICAL[0], edgecolor=viz.SURFACE)
axes[0].set_xscale("log"); axes[0].set_xlabel("Hours from outage start to 90% restored (log)")
axes[0].set_ylabel("Units"); axes[0].set_title("Distribution of restoration times")
kmf = KaplanMeierFitter().fit(s.restore_hours, s.restored)
kmf.plot_survival_function(ax=axes[1], color=viz.CATEGORICAL[0], legend=False)
axes[1].set_xlim(0, 96); axes[1].set_xlabel("Hours since outage start")
axes[1].set_ylabel("Share not yet 90% restored")
axes[1].set_title(f"Kaplan–Meier, all outages (median {kmf.median_survival_time_:.1f} h)")
viz.save(fig, "03_restoration_distribution"); fig

In [ ]:
s.groupby("storm_type")["restore_hours"].describe(percentiles=[.5, .9])[["count", "50%", "90%"]].round(1).sort_values("50%")

## 4. First look: SVI vs restoration time

Unadjusted — storm intensity is not controlled here (that is notebook 04).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for q, col in zip(["Q1 (least)", "Q2", "Q3", "Q4 (most)"], viz.SVI_COLORS):
    d = s[s.svi_quartile == q]
    KaplanMeierFitter(label=q).fit(d.restore_hours, d.restored).plot_survival_function(
        ax=axes[0], color=col, ci_show=False)
axes[0].set_xlim(0, 72); axes[0].set_xlabel("Hours since outage start")
axes[0].set_ylabel("Share not yet 90% restored"); axes[0].set_title("Restoration by SVI quartile (unadjusted)")
axes[0].legend(title="SVI quartile")
dec = s.assign(svi_decile=pd.qcut(s.svi, 10, labels=False) + 1).groupby("svi_decile")["restore_hours"].median()
axes[1].plot(dec.index, dec.values, marker="o", color=viz.CATEGORICAL[0], markersize=6)
axes[1].set_xlabel("SVI decile (1 = least vulnerable)"); axes[1].set_ylabel("Median hours to 90% restored")
axes[1].set_title("Median restoration time by SVI decile")
viz.save(fig, "04_svi_vs_restoration"); fig

## 5. Case studies

In [ ]:
def case_plot(name, states, title, fname):
    h = pd.read_parquet(C.INTERIM / f"case_{name}.parquet")
    h = h[(h.fips // 1000).isin(states)]
    tot = h.groupby("hour")["out"].sum() / 1e6
    peak = h.groupby("fips")["out"].max().rename("peak").reset_index()
    mcc = pd.read_csv(C.RAW / "eaglei" / "MCC.csv", encoding="utf-8-sig")
    mcc = mcc[pd.to_numeric(mcc.County_FIPS, errors="coerce").notna()].astype({"County_FIPS": int})
    peak = peak.merge(mcc, left_on="fips", right_on="County_FIPS")
    peak["peak_pct"] = (100 * peak.peak / peak.Customers).clip(upper=100)
    st = counties[counties.STATEFP.astype(int).isin(states)].merge(peak, on="fips", how="left")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), gridspec_kw={"width_ratios": [1.3, 1]})
    axes[0].plot(tot.index, tot.values, color=viz.CATEGORICAL[0])
    axes[0].fill_between(tot.index, tot.values, color=viz.CATEGORICAL[0], alpha=0.12)
    i = tot.idxmax()
    axes[0].annotate(f"peak {tot.max():.2f}M", (i, tot.max()), xytext=(10, -5),
                     textcoords="offset points", color=viz.INK_2)
    axes[0].set_ylabel("Customers out (millions)"); axes[0].set_title(f"{title}: customers out (UTC)")
    axes[0].tick_params(axis="x", rotation=30)
    st.plot(column="peak_pct", ax=axes[1], cmap=viz.SEQUENTIAL_CMAP, vmin=0, vmax=100, legend=True,
            legend_kwds={"label": "Peak % of customers out", "shrink": 0.7},
            missing_kwds={"color": viz.NO_DATA, "hatch": "////"}, edgecolor=viz.SURFACE, linewidth=0.2)
    axes[1].set_axis_off(); axes[1].set_title("Peak % out by county")
    viz.save(fig, fname); return fig

case_plot("texas_feb2021", [48], "Texas winter storm, Feb 2021", "05a_case_texas_2021")

In [ ]:
case_plot("ida_aug2021", [22, 28], "Hurricane Ida, Aug–Sep 2021", "05b_case_ida_2021")

In [ ]:
# the matched (episode, county) units behind the case studies
cases = df[((df.fips // 1000 == 48) & (df.begin.between("2021-02-10", "2021-02-25"))) |
           ((df.fips // 1000).isin([22, 28]) & (df.begin.between("2021-08-28", "2021-09-15")))]
cases.sort_values("peak", ascending=False)[["state", "fips", "storm_type", "begin", "peak", "peak_pct",
                                            "restore_hours", "restored", "svi"]].head(15).round(3)